# 01 - Data Exploration: MovieLens

Notebook ini memeriksa bahan mentah sebelum masuk ke PostgreSQL.

**Yang dicek:**
- bentuk dan tipe data tiga file CSV;
- missing values dan duplikat;
- parsing tahun dari judul;
- genre per film;
- coverage `tmdbId` dari `links.csv`;
- statistik `ratings.csv`.

**Batasan:** notebook ini hanya membaca CSV. Tidak ada penulisan ke database dan tidak ada panggilan TMDB.

## 1. Setup

In [1]:
from pathlib import Path
import re

import pandas as pd

CWD = Path.cwd()
ROOT_DIR = CWD if (CWD / "data").exists() else CWD.parent
DATA_DIR = ROOT_DIR / "data"

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)

print(f"pandas: {pd.__version__}")
print(f"data directory: {DATA_DIR}")
print(f"csv files: {sorted(path.name for path in DATA_DIR.glob('*.csv'))}")

assert DATA_DIR.exists(), f"data directory not found: {DATA_DIR}"

pandas: 3.1.1
data directory: D:\Project Vidflix\data
csv files: ['links.csv', 'movies.csv', 'ratings.csv', 'tags.csv']


## 2. Load the CSV files

In [2]:
movies = pd.read_csv(DATA_DIR / "movies.csv")
links = pd.read_csv(DATA_DIR / "links.csv")
ratings = pd.read_csv(DATA_DIR / "ratings.csv")

print("movies :", movies.shape)
print("links  :", links.shape)
print("ratings:", ratings.shape)

movies : (9742, 3)
links  : (9742, 3)
ratings: (100836, 4)


### Preview

In [3]:
movies.head(10)

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy
5,6,Heat (1995),Action|Crime|Thriller
6,7,Sabrina (1995),Comedy|Romance
7,8,Tom and Huck (1995),Adventure|Children
8,9,Sudden Death (1995),Action
9,10,GoldenEye (1995),Action|Adventure|Thriller


In [4]:
links.head(10)

,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0
3,4,114885,31357.0
4,5,113041,11862.0
5,6,113277,949.0
6,7,114319,11860.0
7,8,112302,45325.0
8,9,114576,9091.0
9,10,113189,710.0


In [5]:
ratings.head(10)

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931
5,1,70,3.0,964982400
6,1,101,5.0,964980868
7,1,110,4.0,964982176
8,1,151,5.0,964984041
9,1,157,5.0,964984100


## 3. Data types and missing values

In [6]:
for name, df in [("movies", movies), ("links", links), ("ratings", ratings)]:
    print(f"\n{name} - dtypes")
    print(df.dtypes.to_string())
    print(f"{name} - missing values")
    print(df.isna().sum().to_string())


movies - dtypes
movieId    int64
title        str
genres       str
movies - missing values
movieId    0
title      0
genres     0

links - dtypes
movieId      int64
imdbId       int64
tmdbId     float64
links - missing values
movieId    0
imdbId     0
tmdbId     8

ratings - dtypes
userId         int64
movieId        int64
rating       float64
timestamp      int64
ratings - missing values
userId       0
movieId      0
rating       0
timestamp    0


### Duplicates

In [7]:
print("duplicate movieId in movies:", int(movies["movieId"].duplicated().sum()))
print("duplicate movieId in links :", int(links["movieId"].duplicated().sum()))
print(
    "duplicate (userId, movieId) in ratings:",
    int(ratings.duplicated(subset=["userId", "movieId"]).sum()),
)

duplicate movieId in movies: 0
duplicate movieId in links : 0
duplicate (userId, movieId) in ratings: 0


## 4. Parse title and year

MovieLens menyimpan tahun di dalam judul, misalnya `Toy Story (1995)`.
Skema database memisahkan `title` dan `year`, jadi tahun diambil dari akhir judul.

Catatan data: sebagian judul punya spasi ekstra di akhir, misalnya
`'Justice League: The New Frontier (2008) '`. Karena itu string di-`strip()`
dulu sebelum dicocokkan, supaya regex `$` tetap menempel tepat setelah `)`.

In [8]:
YEAR_PATTERN = re.compile(r"^(?P<title>.*)\s+\((?P<year>\d{4})\)$")


def split_title_year(raw_title: str) -> tuple[str, int | None]:
    normalized = raw_title.strip()
    match = YEAR_PATTERN.match(normalized)
    if match is None:
        return normalized, None
    return match.group("title").strip(), int(match.group("year"))


parsed_titles = movies["title"].apply(split_title_year)

movies_clean = movies.copy()
movies_clean["title"] = [item[0] for item in parsed_titles]
movies_clean["year"] = pd.array([item[1] for item in parsed_titles], dtype="Int64")

print("movies without year:", int(movies_clean["year"].isna().sum()))
movies_clean[["movieId", "title", "year"]].head(10)

movies without year: 13


,movieId,title,year
0,1,Toy Story,1995
1,2,Jumanji,1995
2,3,Grumpier Old Men,1995
3,4,Waiting to Exhale,1995
4,5,Father of the Bride Part II,1995
5,6,Heat,1995
6,7,Sabrina,1995
7,8,Tom and Huck,1995
8,9,Sudden Death,1995
9,10,GoldenEye,1995


In [9]:
still_has_year = movies_clean["title"].str.contains(r"\(\d{4}\)\s*$", regex=True).sum()
trailing_space = (movies_clean["title"] != movies_clean["title"].str.strip()).sum()
print("titles still containing a year:", int(still_has_year))
print("titles with trailing whitespace:", int(trailing_space))

movies_clean[movies_clean["year"].isna()][["movieId", "title"]].head(20)

titles still containing a year: 0
titles with trailing whitespace: 0


,movieId,title
6059,40697,Babylon 5
9031,140956,Ready Player One
9091,143410,Hyena Road
9138,147250,The Adventures of Sherlock Holmes and Doctor W...
9179,149334,Nocturnal Animals
9259,156605,Paterson
9367,162414,Moonlight
9448,167570,The OA
9514,171495,Cosmos
9515,171631,Maria Bamford: Old Baby


## 5. Genres

In [10]:
NO_GENRES = "(no genres listed)"


def split_genres(raw_genres: str) -> list[str]:
    if raw_genres == NO_GENRES:
        return []
    return raw_genres.split("|")


movies_clean["genres_list"] = movies_clean["genres"].apply(split_genres)

print("movies without genres:", int((movies_clean["genres_list"].str.len() == 0).sum()))
movies_clean[["movieId", "title", "genres", "genres_list"]].head(10)

movies without genres: 34


,movieId,title,genres,genres_list
0,1,Toy Story,Adventure|Animation|Children|Comedy|Fantasy,"[Adventure, Animation, Children, Comedy, Fantasy]"
1,2,Jumanji,Adventure|Children|Fantasy,"[Adventure, Children, Fantasy]"
2,3,Grumpier Old Men,Comedy|Romance,"[Comedy, Romance]"
3,4,Waiting to Exhale,Comedy|Drama|Romance,"[Comedy, Drama, Romance]"
4,5,Father of the Bride Part II,Comedy,[Comedy]
5,6,Heat,Action|Crime|Thriller,"[Action, Crime, Thriller]"
6,7,Sabrina,Comedy|Romance,"[Comedy, Romance]"
7,8,Tom and Huck,Adventure|Children,"[Adventure, Children]"
8,9,Sudden Death,Action,[Action]
9,10,GoldenEye,Action|Adventure|Thriller,"[Action, Adventure, Thriller]"


In [11]:
genre_counts = (
    movies_clean.explode("genres_list")["genres_list"]
    .value_counts()
    .to_frame("movie_count")
)
print("unique genres:", len(genre_counts))
genre_counts

unique genres: 19


,movie_count
genres_list,
Drama,4361
Comedy,3756
Thriller,1894
Action,1828
Romance,1596
Adventure,1263
Crime,1199
Sci-Fi,980
Horror,978


In [12]:
movies_clean["genres_list"].str.len().value_counts().sort_index().to_frame("movie_count")

,movie_count
genres_list,
0,34
1,2817
2,3218
3,2338
4,987
5,271
6,63
7,12
8,1


## 6. TMDB coverage

In [13]:
links_clean = links.copy()
links_clean["tmdbId"] = links_clean["tmdbId"].astype("Int64")

movies_with_links = movies_clean.merge(
    links_clean[["movieId", "tmdbId"]],
    on="movieId",
    how="left",
)

print("movies:", len(movies_with_links))
print("movies with tmdbId:", int(movies_with_links["tmdbId"].notna().sum()))
print("movies without tmdbId:", int(movies_with_links["tmdbId"].isna().sum()))
movies_with_links[["movieId", "title", "year", "tmdbId"]].head(10)

movies: 9742
movies with tmdbId: 9734
movies without tmdbId: 8


,movieId,title,year,tmdbId
0,1,Toy Story,1995,862
1,2,Jumanji,1995,8844
2,3,Grumpier Old Men,1995,15602
3,4,Waiting to Exhale,1995,31357
4,5,Father of the Bride Part II,1995,11862
5,6,Heat,1995,949
6,7,Sabrina,1995,11860
7,8,Tom and Huck,1995,45325
8,9,Sudden Death,1995,9091
9,10,GoldenEye,1995,710


In [14]:
print(
    "links without matching movie:",
    int((~links["movieId"].isin(movies["movieId"])).sum()),
)
movies_with_links[movies_with_links["tmdbId"].isna()][["movieId", "title", "year"]].head(20)

links without matching movie: 0


,movieId,title,year
624,791,"Last Klezmer: Leopold Kozlowski, His Life and ...",1994
843,1107,Loser,1991
2141,2851,Saturn 3,1980
3027,4051,Horrors of Spider Island (Ein Toter Hing im Netz),1960
5532,26587,"Decalogue, The (Dekalog)",1989
5854,32600,Eros,2004
6059,40697,Babylon 5,<NA>
7382,79299,"No. 1 Ladies' Detective Agency, The",2008


## 7. MovieLens ratings

In [15]:
print("ratings:", len(ratings))
print("unique users:", ratings["userId"].nunique())
print("unique movies:", ratings["movieId"].nunique())
print("rating range:", ratings["rating"].min(), "to", ratings["rating"].max())
ratings["rating"].value_counts().sort_index().to_frame("count")

ratings: 100836
unique users: 610
unique movies: 9724
rating range: 0.5 to 5.0


,count
rating,
0.5,1370
1.0,2811
1.5,1791
2.0,7551
2.5,5550
3.0,20047
3.5,13136
4.0,26818
4.5,8551


In [16]:
possible_pairs = ratings["userId"].nunique() * movies["movieId"].nunique()
print(f"sparsity: {len(ratings) / possible_pairs:.4%}")

rated_movie_ids = set(ratings["movieId"])
known_movie_ids = set(movies["movieId"])
print("ratings for unknown movieId:", len(rated_movie_ids - known_movie_ids))
print("movies with at least one rating:", len(rated_movie_ids & known_movie_ids))

sparsity: 1.6968%
ratings for unknown movieId: 0
movies with at least one rating: 9724


## 8. Import summary

In [17]:
summary = pd.DataFrame(
    {
        "metric": [
            "movies in CSV",
            "movies with tmdbId",
            "movies without tmdbId",
            "movies without year",
            "movies without genres",
            "ratings in CSV",
            "unique users",
            "unique rated movies",
        ],
        "value": [
            len(movies_clean),
            int(movies_with_links["tmdbId"].notna().sum()),
            int(movies_with_links["tmdbId"].isna().sum()),
            int(movies_clean["year"].isna().sum()),
            int((movies_clean["genres_list"].str.len() == 0).sum()),
            len(ratings),
            int(ratings["userId"].nunique()),
            int(ratings["movieId"].nunique()),
        ],
    }
)
summary

,metric,value
0,movies in CSV,9742
1,movies with tmdbId,9734
2,movies without tmdbId,8
3,movies without year,13
4,movies without genres,34
5,ratings in CSV,100836
6,unique users,610
7,unique rated movies,9724


## 9. Decisions for the import script

- `title` dan `year` dipisah; `year` boleh kosong.
- `genres` disimpan sebagai array; film tanpa genre menjadi array kosong.
- Film tanpa `tmdbId` tetap diimpor; `overview` dan `poster_path` dibiarkan kosong.
- `ratings.csv` diimpor ke `movielens_ratings`; `timestamp` tidak dipakai.
- Film dengan `tmdbId` akan di-enrich lewat TMDB pada script import.
- Setelah import, cek coverage `overview` dan `poster_path` sebelum Tahap 3.